In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics

In [ ]:
import ultralytics

print("Ultralytics version:", ultralytics.__version__)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully!")

In [ ]:
results = model("road.jpg")

print("Detection completed!")

In [ ]:
results[0].show()

In [ ]:
for result in results:

    print("Detected Objects:")
    print("-" * 40)

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        class_name = model.names[class_id]

        print(f"{class_name:15} : {confidence:.2f}")

In [ ]:
from collections import Counter, defaultdict

object_counts = Counter()
object_confidences = defaultdict(list)

for result in results:
    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        class_name = model.names[class_id]

        object_counts[class_name] += 1
        object_confidences[class_name].append(confidence)

print("OBJECT DETECTION SUMMARY")
print("=" * 50)

for object_name in object_counts:
    count = object_counts[object_name]
    avg_confidence = sum(object_confidences[object_name]) / count

    print(
        f"{object_name:20} "
        f"Count: {count:3}   "
        f"Average Confidence: {avg_confidence:.2%}"
    )

In [ ]:
total_objects = sum(object_counts.values())

all_confidences = []

for confidence_list in object_confidences.values():
    all_confidences.extend(confidence_list)

average_confidence = sum(all_confidences) / len(all_confidences)

print("OVERALL DETECTION STATISTICS")
print("=" * 50)
print(f"Total objects detected : {total_objects}")
print(f"Average confidence     : {average_confidence:.2%}")

In [ ]:
import time

for _ in range(3):
    model("road.jpg", verbose=False)

times = []

for _ in range(20):
    start = time.perf_counter()

    model("road.jpg", verbose=False)

    end = time.perf_counter()
    times.append(end - start)

average_time = sum(times) / len(times)
fps = 1 / average_time

print("PERFORMANCE RESULTS")
print("=" * 50)
print(f"Average inference time : {average_time:.4f} seconds")
print(f"Average FPS            : {fps:.2f}")

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

In [ ]:
results_timing = model("road.jpg", verbose=False)

speed = results_timing[0].speed

print("YOLO TIMING")
print("=" * 50)
print(f"Preprocessing : {speed['preprocess']:.2f} ms")
print(f"Inference     : {speed['inference']:.2f} ms")
print(f"Postprocess   : {speed['postprocess']:.2f} ms")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import shutil

os.makedirs("test_images", exist_ok=True)

for i in range(1, 15):
    filename = f"road{i}.jpg"

    if os.path.exists(filename):
        shutil.move(filename, os.path.join("test_images", filename))

print("Images organized successfully!")
print("Number of test images:", len(os.listdir("test_images")))
print(os.listdir("test_images"))

In [ ]:
import os
import shutil

if os.path.exists("road2.jpg"):
    shutil.move("road2.jpg", "test_images/road2.jpg")
    print("Moved road2.jpg")

if os.path.exists("road 2.jpg"):
    shutil.move("road 2.jpg", "test_images/road 2.jpg")
    print("Moved road 2.jpg")

print("\nFiles inside test_images:")
for filename in sorted(os.listdir("test_images")):
    print(filename)

print("\nTotal images:", len(os.listdir("test_images")))

In [ ]:
import os
import time
from collections import Counter

image_folder = "test_images"

all_results = []

image_files = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

for image_name in sorted(image_files):

    image_path = os.path.join(image_folder, image_name)

    start_time = time.perf_counter()

    results = model(image_path, verbose=False)

    end_time = time.perf_counter()

    inference_time = end_time - start_time

    object_counts = Counter()
    confidences = []

    for result in results:

        for box in result.boxes:

            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            class_name = model.names[class_id]

            object_counts[class_name] += 1
            confidences.append(confidence)

    total_objects = sum(object_counts.values())

    average_confidence = (
        sum(confidences) / len(confidences)
        if confidences else 0
    )

    fps = 1 / inference_time

    all_results.append({
        "image": image_name,
        "total_objects": total_objects,
        "average_confidence": average_confidence,
        "inference_time": inference_time,
        "fps": fps,
        "object_counts": dict(object_counts)
    })

    print(
        f"{image_name:15} | "
        f"Objects: {total_objects:3} | "
        f"Confidence: {average_confidence:.2%} | "
        f"FPS: {fps:.2f}"
    )

print("\nExperiment completed!")
print("Images analyzed:", len(all_results))

In [ ]:
import pandas as pd

summary_data = []

for result in all_results:

    summary_data.append({
        "Image": result["image"],
        "Total Objects": result["total_objects"],
        "Average Confidence": result["average_confidence"],
        "Inference Time (s)": result["inference_time"],
        "FPS": result["fps"]
    })

df = pd.DataFrame(summary_data)

df

In [ ]:
overall_objects = df["Total Objects"].sum()
overall_confidence = df["Average Confidence"].mean()
average_inference = df["Inference Time (s)"].mean()
average_fps = df["FPS"].mean()

print("OVERALL EXPERIMENT RESULTS")
print("=" * 55)

print(f"Images analyzed           : {len(df)}")
print(f"Total objects detected    : {overall_objects}")
print(f"Average confidence       : {overall_confidence:.2%}")
print(f"Average inference time   : {average_inference:.4f} seconds")
print(f"Average FPS              : {average_fps:.2f}")

In [ ]:
df.to_csv("yolo_detection_results.csv", index=False)

print("Results saved as yolo_detection_results.csv")

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import shutil

os.makedirs("test_images", exist_ok=True)

for filename in uploaded.keys():
    destination = os.path.join("test_images", filename)

    if not os.path.exists(destination):
        shutil.move(filename, destination)

print("Images organized successfully!")
print("Number of images:", len(os.listdir("test_images")))

print("\nFiles:")
for filename in sorted(os.listdir("test_images")):
    print(filename)

In [ ]:
import os

print("Folder exists:", os.path.exists("test_images"))
print("Number of images:", len(os.listdir("test_images")))

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully!")

In [ ]:
import os

print("Model loaded:", model is not None)
print("Images:", len(os.listdir("test_images")))

In [ ]:
import os

print("Model loaded:", model is not None)
print("Images:", len(os.listdir("test_images")))

In [ ]:
import os
import time
from collections import Counter

image_folder = "test_images"

all_results = []

image_files = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("MULTI-IMAGE YOLO EXPERIMENT")
print("=" * 70)
print("Images found:", len(image_files))
print()

for image_name in sorted(image_files):

    image_path = os.path.join(image_folder, image_name)

    start_time = time.perf_counter()

    results = model(image_path, verbose=False)

    end_time = time.perf_counter()

    inference_time = end_time - start_time

    object_counts = Counter()
    confidences = []

    for result in results:

        for box in result.boxes:

            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            class_name = model.names[class_id]

            object_counts[class_name] += 1
            confidences.append(confidence)

    total_objects = sum(object_counts.values())

    average_confidence = (
        sum(confidences) / len(confidences)
        if confidences else 0
    )

    fps = 1 / inference_time

    all_results.append({
        "image": image_name,
        "total_objects": total_objects,
        "average_confidence": average_confidence,
        "inference_time": inference_time,
        "fps": fps,
        "object_counts": dict(object_counts)
    })

    print(
        f"{image_name:12} | "
        f"Objects: {total_objects:3} | "
        f"Confidence: {average_confidence:.2%} | "
        f"Time: {inference_time:.4f}s | "
        f"FPS: {fps:.2f}"
    )

print()
print("=" * 70)
print("EXPERIMENT COMPLETED")
print("Images analyzed:", len(all_results))

In [ ]:
import pandas as pd

summary_data = []

for result in all_results:
    summary_data.append({
        "Image": result["image"],
        "Total Objects": result["total_objects"],
        "Average Confidence": result["average_confidence"],
        "Inference Time (s)": result["inference_time"],
        "FPS": result["fps"]
    })

df = pd.DataFrame(summary_data)

df

In [ ]:
overall_objects = df["Total Objects"].sum()
overall_confidence = df["Average Confidence"].mean()
average_inference = df["Inference Time (s)"].mean()
average_fps = df["FPS"].mean()

print("OVERALL EXPERIMENT RESULTS")
print("=" * 55)

print(f"Images analyzed        : {len(df)}")
print(f"Total objects detected : {overall_objects}")
print(f"Average confidence     : {overall_confidence:.2%}")
print(f"Average inference time : {average_inference:.4f} seconds")
print(f"Average FPS            : {average_fps:.2f}")

In [ ]:
df.to_csv("yolo_detection_results.csv", index=False)

print("Results saved successfully!")

In [ ]:
import os

print("CSV exists:", os.path.exists("yolo_detection_results.csv"))

In [ ]:
from google.colab import files

files.download("yolo_detection_results.csv")

In [ ]:
from collections import Counter

total_object_counts = Counter()

for result in all_results:
    total_object_counts.update(result["object_counts"])

print("OBJECT-WISE DETECTION SUMMARY")
print("=" * 50)

for object_name, count in total_object_counts.most_common():
    print(f"{object_name:20} : {count}")

In [ ]:
object_df = pd.DataFrame(
    total_object_counts.items(),
    columns=["Object", "Detections"]
)

object_df = object_df.sort_values(
    "Detections",
    ascending=False
).reset_index(drop=True)

object_df

In [ ]:
object_confidences = {}

for result in all_results:

    image_path = os.path.join(
        "test_images",
        result["image"]
    )

    results = model(image_path, verbose=False)

    for detection in results:

        for box in detection.boxes:

            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            class_name = model.names[class_id]

            if class_name not in object_confidences:
                object_confidences[class_name] = []

            object_confidences[class_name].append(confidence)

object_analysis = []

for object_name, confidence_list in object_confidences.items():

    object_analysis.append({
        "Object": object_name,
        "Detections": len(confidence_list),
        "Average Confidence": sum(confidence_list) / len(confidence_list)
    })

object_analysis_df = pd.DataFrame(object_analysis)

object_analysis_df = object_analysis_df.sort_values(
    "Detections",
    ascending=False
).reset_index(drop=True)

object_analysis_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.bar(
    object_df["Object"],
    object_df["Detections"]
)

plt.xlabel("Object Class")
plt.ylabel("Number of Detections")
plt.title("YOLO Object Detection Distribution")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.bar(
    object_analysis_df["Object"],
    object_analysis_df["Average Confidence"]
)

plt.xlabel("Object Class")
plt.ylabel("Average Confidence")
plt.title("Average YOLO Detection Confidence by Object Class")

plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.tight_layout()

plt.show()

In [ ]:
confidence_percentage_df = object_analysis_df.copy()

confidence_percentage_df["Average Confidence (%)"] = (
    confidence_percentage_df["Average Confidence"] * 100
)

confidence_percentage_df[
    ["Object", "Detections", "Average Confidence (%)"]
]

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    confidence_percentage_df["Object"],
    confidence_percentage_df["Average Confidence (%)"]
)

plt.xlabel("Object Class")
plt.ylabel("Average Confidence (%)")
plt.title("Average YOLO Detection Confidence")

plt.xticks(rotation=45)
plt.ylim(0, 100)
plt.tight_layout()

plt.show()

In [ ]:
final_object_analysis = confidence_percentage_df[
    ["Object", "Detections", "Average Confidence (%)"]
].copy()

final_object_analysis

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    df["Image"],
    df["Average Confidence"] * 100,
    marker="o"
)

plt.xlabel("Test Image")
plt.ylabel("Average Confidence (%)")
plt.title("YOLO Detection Confidence Across Test Images")

plt.xticks(rotation=45)
plt.ylim(0, 100)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    df["Image"],
    df["FPS"],
    marker="o"
)

plt.xlabel("Test Image")
plt.ylabel("FPS")
plt.title("YOLO Inference Speed Across Test Images")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
final_object_analysis.to_csv(
    "yolo_object_analysis.csv",
    index=False
)

print("Object analysis saved successfully!")

In [ ]:
df.to_csv(
    "yolo_image_results.csv",
    index=False
)

print("Image-level results saved successfully!")

In [ ]:
df

In [ ]:
final_object_analysis

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

image_folder = "test_images"

image_files = sorted([
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

axes = axes.flatten()

for i, image_name in enumerate(image_files):

    image_path = os.path.join(image_folder, image_name)
    image = Image.open(image_path)

    axes[i].imshow(image)
    axes[i].set_title(image_name)
    axes[i].axis("off")

for i in range(len(image_files), len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
environment_labels = {
    "road1.jpg": "Daylight",
    "road 2.jpg": "Daylight",
    "road2.jpg": "Daylight",
    "road3.jpg": "Daylight",
    "road4.jpg": "Heavy Traffic",
    "road5.jpg": "Heavy Traffic",
    "road6.jpg": "Heavy Traffic",
    "road7.jpg": "Daylight",
    "road8.jpg": "Daylight",
    "road9.jpg": "Pedestrian Scene",
    "road10.jpg": "Pedestrian Scene",
    "road11.jpg": "Night",
    "road12.jpg": "Night",
    "road13.jpg": "Fog",
    "road14.jpg": "Rain"
}

df["Environment"] = df["Image"].map(environment_labels)

df[["Image", "Environment", "Total Objects", "Average Confidence"]]

In [ ]:
print(df[["Image", "Environment"]])

In [ ]:
environment_analysis = (
    df.groupby("Environment")
      .agg(
          Images=("Image", "count"),
          Total_Objects=("Total Objects", "sum"),
          Average_Confidence=("Average Confidence", "mean")
      )
      .reset_index()
)

environment_analysis["Average_Confidence_%"] = (
    environment_analysis["Average_Confidence"] * 100
)

environment_analysis

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.bar(
    environment_analysis["Environment"],
    environment_analysis["Average_Confidence_%"]
)

plt.xlabel("Environmental Condition")
plt.ylabel("Average Confidence (%)")
plt.title("YOLO Performance Under Different Environmental Conditions")

plt.xticks(rotation=30)
plt.ylim(0, 100)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    environment_analysis["Environment"],
    environment_analysis["Total_Objects"]
)

plt.xlabel("Environmental Condition")
plt.ylabel("Total Objects Detected")
plt.title("Object Detection Across Environmental Conditions")

plt.xticks(rotation=30)
plt.tight_layout()

plt.show()

In [ ]:
environment_analysis.to_csv(
    "yolo_environment_analysis.csv",
    index=False
)

print("Environmental analysis saved successfully!")

In [ ]:
import os

print(
    "File exists:",
    os.path.exists("yolo_environment_analysis.csv")
)

In [ ]:
import time
import torch

test_image = "test_images/road1.jpg"

print("Warming up GPU...")

for _ in range(10):
    model(test_image, verbose=False)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("Warm-up completed.")

times = []

for _ in range(50):

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    model(test_image, verbose=False)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_time = time.perf_counter()

    times.append(end_time - start_time)

average_time = sum(times) / len(times)
fps = 1 / average_time

print("\nCONTROLLED REAL-TIME PERFORMANCE")
print("=" * 50)
print(f"Number of runs       : {len(times)}")
print(f"Average inference    : {average_time:.4f} seconds")
print(f"Average inference    : {average_time * 1000:.2f} ms")
print(f"Estimated FPS        : {fps:.2f}")

In [ ]:
import time
import torch

test_images = [
    ("Daylight", "test_images/road1.jpg"),
    ("Heavy Traffic", "test_images/road5.jpg"),
    ("Night", "test_images/road11.jpg"),
    ("Fog", "test_images/road13.jpg"),
    ("Rain", "test_images/road14.jpg")
]

fps_results = []

for condition, image_path in test_images:

    for _ in range(5):
        model(image_path, verbose=False)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []

    for _ in range(20):

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        model(image_path, verbose=False)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()

        times.append(end - start)

    average_time = sum(times) / len(times)
    fps = 1 / average_time

    fps_results.append({
        "Condition": condition,
        "Average Inference (ms)": average_time * 1000,
        "FPS": fps
    })

    print(
        f"{condition:15} | "
        f"Time: {average_time * 1000:.2f} ms | "
        f"FPS: {fps:.2f}"
    )

In [ ]:
fps_df = pd.DataFrame(fps_results)

fps_df

In [ ]:
fps_df.to_csv(
    "yolo_realtime_performance.csv",
    index=False
)

print("Real-time performance results saved!")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.bar(
    fps_df["Condition"],
    fps_df["FPS"]
)

plt.xlabel("Environmental Condition")
plt.ylabel("Frames Per Second (FPS)")
plt.title("YOLO11n Real-Time Performance")

plt.xticks(rotation=30)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    fps_df["Condition"],
    fps_df["Average Inference (ms)"]
)

plt.xlabel("Environmental Condition")
plt.ylabel("Average Inference Time (ms)")
plt.title("YOLO11n Inference Time Under Different Conditions")

plt.xticks(rotation=30)
plt.tight_layout()

plt.show()

In [ ]:
from ultralytics import YOLO

models = {
    "YOLO11n": YOLO("yolo11n.pt"),
    "YOLO11s": YOLO("yolo11s.pt"),
    "YOLO11m": YOLO("yolo11m.pt")
}

print("All models loaded successfully!")

In [ ]:
import os
import time
import pandas as pd

comparison_images = [
    "road1.jpg",
    "road5.jpg",
    "road11.jpg",
    "road13.jpg",
    "road14.jpg"
]

comparison_results = []

for model_name, current_model in models.items():

    print(f"\nTesting {model_name}...")

    for _ in range(5):
        current_model(
            os.path.join("test_images", comparison_images[0]),
            verbose=False
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    for image_name in comparison_images:

        image_path = os.path.join(
            "test_images",
            image_name
        )

        times = []

        for _ in range(10):

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start = time.perf_counter()

            results = current_model(
                image_path,
                verbose=False
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            end = time.perf_counter()

            times.append(end - start)

        average_time = sum(times) / len(times)
        fps = 1 / average_time

        confidences = []
        object_count = 0

        for result in results:

            object_count += len(result.boxes)

            for box in result.boxes:
                confidences.append(float(box.conf[0]))

        average_confidence = (
            sum(confidences) / len(confidences)
            if confidences else 0
        )

        comparison_results.append({
            "Model": model_name,
            "Image": image_name,
            "Objects": object_count,
            "Average Confidence": average_confidence,
            "Inference Time (ms)": average_time * 1000,
            "FPS": fps
        })

        print(
            f"{image_name:12} | "
            f"Objects: {object_count:3} | "
            f"Confidence: {average_confidence:.2%} | "
            f"FPS: {fps:.2f}"
        )

print("\nComparison completed!")

In [ ]:
comparison_df = pd.DataFrame(comparison_results)

model_summary = (
    comparison_df
    .groupby("Model")
    .agg(
        Average_Objects=("Objects", "mean"),
        Average_Confidence=("Average Confidence", "mean"),
        Average_Inference_ms=("Inference Time (ms)", "mean"),
        Average_FPS=("FPS", "mean")
    )
    .reset_index()
)

model_summary

In [ ]:
model_summary.to_csv(
    "yolo_model_comparison.csv",
    index=False
)

comparison_df.to_csv(
    "yolo_model_detailed_comparison.csv",
    index=False
)

print("Model comparison files saved successfully!")

In [ ]:
plt.figure(figsize=(9, 6))

plt.bar(
    model_summary["Model"],
    model_summary["Average_FPS"]
)

plt.xlabel("YOLO Model")
plt.ylabel("Average FPS")
plt.title("Real-Time Performance Comparison of YOLO Models")

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

plt.bar(
    model_summary["Model"],
    model_summary["Average_Inference_ms"]
)

plt.xlabel("YOLO Model")
plt.ylabel("Average Inference Time (ms)")
plt.title("Inference Time Comparison of YOLO Models")

plt.tight_layout()

plt.show()

In [ ]:
from google.colab import files

uploaded_video = files.upload()

In [ ]:
import os

video_files = [
    f for f in os.listdir("/content")
    if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
]

print("Video files found:")
for f in video_files:
    print(f)

In [ ]:
import os

video_files = [
    f for f in os.listdir("/content")
    if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
]

print("Video files found:")

for i, f in enumerate(video_files):
    print(i, ":", f)

video_path = os.path.join("/content", video_files[0])

print("\nSelected video:")
print(video_path)

In [ ]:
tracking_results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    conf=0.25,
    save=True,
    verbose=False
)

print("Video tracking completed!")

In [ ]:
import glob
import os

video_outputs = glob.glob(
    "/content/runs/**/*.mp4",
    recursive=True
)

print("Processed video files:")
print("=" * 60)

for video in video_outputs:
    print(video)
    print("Size:", round(os.path.getsize(video) / (1024 * 1024), 2), "MB")

In [ ]:
from IPython.display import Video, display

if video_outputs:
    output_video = video_outputs[-1]

    print("Selected output:")
    print(output_video)

    display(Video(output_video, embed=True))
else:
    print("No MP4 output found.")

In [ ]:
import os

output_dir = "/content/runs/detect/predict"

print("Folder exists:", os.path.exists(output_dir))

if os.path.exists(output_dir):
    print("\nFiles inside output folder:")
    for f in os.listdir(output_dir):
        path = os.path.join(output_dir, f)
        print(f, "->", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

In [ ]:
from IPython.display import Video, display

output_video = "/content/runs/detect/predict/Traffic sound, Indian Traffic Sounds, Traffic Noise, Vehicle Noise, Road traffic#Shorts#Viral #cars - 720.avi"

display(Video(output_video, embed=True))

In [ ]:
import cv2
import os

input_video = "/content/runs/detect/predict/Traffic sound, Indian Traffic Sounds, Traffic Noise, Vehicle Noise, Road traffic#Shorts#Viral #cars - 720.avi"

output_mp4 = "/content/YOLO_traffic_detection.mp4"

cap = cv2.VideoCapture(input_video)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_mp4,
    fourcc,
    fps,
    (width, height)
)

while True:
    ret, frame = cap.read()

    if not ret:
        break

    out.write(frame)

cap.release()
out.release()

print("MP4 conversion completed!")
print("Output:", output_mp4)
print("Exists:", os.path.exists(output_mp4))

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/YOLO_traffic_detection.mp4",
        embed=True
    )
)

In [ ]:
import subprocess
import os

input_video = "/content/YOLO_traffic_detection.mp4"
output_h264 = "/content/YOLO_traffic_detection_H264.mp4"

command = [
    "ffmpeg",
    "-y",
    "-i", input_video,
    "-c:v", "libx264",
    "-preset", "fast",
    "-crf", "23",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    output_h264
]

result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("Conversion completed:", os.path.exists(output_h264))

if os.path.exists(output_h264):
    print(
        "File size:",
        round(os.path.getsize(output_h264) / (1024 * 1024), 2),
        "MB"
    )
else:
    print(result.stderr[-2000:])

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/YOLO_traffic_detection_H264.mp4",
        embed=True
    )
)

In [ ]:
import pandas as pd

print("==============================================")
print("      FINAL YOLO PROJECT RESULTS")
print("==============================================")

print(f"\nImages analyzed              : {len(df)}")
print(f"Total objects detected       : {df['Total Objects'].sum()}")
print(
    f"Overall average confidence   : "
    f"{df['Average Confidence'].mean() * 100:.2f}%"
)

print(
    f"Average controlled FPS       : "
    f"{fps_df['FPS'].mean():.2f}"
)

print(
    f"Average inference time      : "
    f"{fps_df['Average Inference (ms)'].mean():.2f} ms"
)

print("\nObject Classes Detected:")
for _, row in final_object_analysis.iterrows():
    print(
        f"  {row['Object']:15} : "
        f"{row['Detections']} detections"
    )

In [ ]:
final_summary = pd.DataFrame({
    "Metric": [
        "Test Images",
        "Total Objects Detected",
        "Average Detection Confidence",
        "Average Controlled FPS",
        "Average Inference Time"
    ],
    "Result": [
        len(df),
        df["Total Objects"].sum(),
        f"{df['Average Confidence'].mean() * 100:.2f}%",
        f"{fps_df['FPS'].mean():.2f} FPS",
        f"{fps_df['Average Inference (ms)'].mean():.2f} ms"
    ]
})

final_summary

In [ ]:
final_summary.to_csv(
    "final_project_summary.csv",
    index=False
)

print("Final summary saved!")

In [ ]:
model_summary

In [ ]:
model_summary.to_csv(
    "final_model_comparison.csv",
    index=False
)

print("Final model comparison saved!")

In [ ]:
import cv2

input_video = "/content/YOLO_traffic_detection_H264.mp4"

cap = cv2.VideoCapture(input_video)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

middle_frame = total_frames // 2
cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)

success, frame = cap.read()

if success:
    screenshot_path = "/content/YOLO_detection_sample.jpg"
    cv2.imwrite(screenshot_path, frame)
    print("Screenshot saved:", screenshot_path)

cap.release()

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename="/content/YOLO_detection_sample.jpg"
    )
)


In [ ]:
import os

important_files = [
    "yolo_detection_results.csv",
    "yolo_image_results.csv",
    "yolo_object_analysis.csv",
    "yolo_environment_analysis.csv",
    "yolo_realtime_performance.csv",
    "yolo_model_comparison.csv",
    "yolo_model_detailed_comparison.csv",
    "final_project_summary.csv",
    "final_model_comparison.csv",
    "YOLO_detection_sample.jpg",
    "YOLO_traffic_detection_H264.mp4"
]

print("PROJECT FILE CHECK")
print("=" * 60)

for filename in important_files:
    exists = os.path.exists("/content/" + filename)
    print(f"{'✓' if exists else '✗'} {filename}")

In [ ]:
from google.colab import files

download_files = [
    "final_project_summary.csv",
    "final_model_comparison.csv",
    "yolo_object_analysis.csv",
    "yolo_environment_analysis.csv",
    "yolo_realtime_performance.csv",
    "yolo_model_comparison.csv",
    "YOLO_detection_sample.jpg",
    "YOLO_traffic_detection_H264.mp4"
]

for filename in download_files:
    if os.path.exists("/content/" + filename):
        files.download("/content/" + filename)